# `mutate` — Reference

`mutate` creates or overwrites columns from a `qry()`-style spec string, each entry evaluated in order via pandas `eval()` — a plain formula per column, no lambda required.

| Syntax | Meaning |
|---|---|
| `"new_col: expr"` | one derived column |
| `"a: expr1, b: expr2"` | several in one call (comma-separated) |
| `"'new_col': expr"` | quoting the key is optional, same as `qry()` |
| `"new_col: if_else(cond, true_val, false_val)"` | dplyr-style two-branch conditional |
| `"new_col: case_when(cond1: v1, cond2: v2, default)"` | multi-branch conditional; last bare value is the catch-all |

Column names *inside* the expression must stay unquoted — see the example below.

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [2]:
# Body mass index style ratio — a plain arithmetic formula, no lambda needed
(penguins
 .pt.mutate('bmi: body_mass_g / bill_length_mm ** 2')
 .pt.select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
 .sample(10)
)

,species,body_mass_g,bill_length_mm,bmi
142,Adelie,3050.0,32.1,2.959987
230,Gentoo,4650.0,40.9,2.779754
31,Adelie,3900.0,37.2,2.818245
18,Adelie,3325.0,34.4,2.809796
177,Chinstrap,4150.0,52.0,1.534763
199,Chinstrap,4300.0,49.0,1.790920
26,Adelie,3550.0,40.6,2.153656
296,Gentoo,4600.0,47.5,2.038781
2,Adelie,3250.0,40.3,2.001121
133,Adelie,4475.0,37.5,3.182222


In [3]:
# Column names inside the expression must stay unquoted — quoting one turns it
# into a string literal, not a column reference, and breaks the arithmetic
try:
    penguins.pt.mutate("bmi: 'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as e:
    print('TypeError:', e)

TypeError: unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call
(penguins
 .pt.mutate('heavy: body_mass_g > 4000, mass_kg: body_mass_g / 1000')
 .pt.select('species', 'body_mass_g', 'mass_kg', 'heavy')
 .sample(10)
)

,species,body_mass_g,mass_kg,heavy
302,Gentoo,4725.0,4.725,True
272,Gentoo,4400.0,4.400,True
255,Gentoo,5400.0,5.400,True
126,Adelie,3275.0,3.275,False
106,Adelie,3750.0,3.750,False
28,Adelie,3150.0,3.150,False
64,Adelie,2850.0,2.850,False
62,Adelie,3600.0,3.600,False
201,Chinstrap,3675.0,3.675,False
229,Gentoo,5150.0,5.150,True


In [5]:
# mass_lb references mass_kg, derived by the entry just before it
(penguins
 .pt.mutate('mass_kg: body_mass_g / 1000, mass_lb: mass_kg * 2.20462')
 .pt.select('species', 'mass_kg', 'mass_lb')
 .sample(10)
)

,species,mass_kg,mass_lb
222,Gentoo,4.45,9.810559
153,Chinstrap,3.90,8.598018
3,Adelie,NaN,NaN
95,Adelie,4.30,9.479866
121,Adelie,3.50,7.716170
167,Chinstrap,4.05,8.928711
4,Adelie,3.45,7.605939
97,Adelie,4.35,9.590097
329,Gentoo,5.50,12.125410
284,Gentoo,4.70,10.361714


## String comparisons, optional key quoting, and local variables
Quoting the key (`'is_adelie'` vs `is_adelie`) is optional, same as `qry()`. String literals *inside* the expression (e.g. `'Adelie'`) still need real quotes — only the column names must stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.

In [6]:
unquoted = penguins.pt.mutate("is_adelie: species == 'Adelie'")
quoted = penguins.pt.mutate("'is_adelie': species == 'Adelie'")
(unquoted['is_adelie'] == quoted['is_adelie']).all()

np.True_

In [7]:
# @-prefixed names resolve against the scope that called mutate(), not mutate()'s own internals
threshold = 4000
(penguins
 .pt.mutate('heavy: body_mass_g >= @threshold')
 .pt.select('species', 'body_mass_g', 'heavy')
 .sample(10)
)

,species,body_mass_g,heavy
100,Adelie,3725.0,False
341,Gentoo,5750.0,True
128,Adelie,3050.0,False
203,Chinstrap,3950.0,False
264,Gentoo,5550.0,True
0,Adelie,3750.0,False
244,Gentoo,5000.0,True
33,Adelie,3900.0,False
281,Gentoo,5300.0,True
75,Adelie,4250.0,True


## Overwriting an existing column

In [8]:
# mutate() can overwrite a column in place, e.g. converting units
(penguins
 .pt.mutate('body_mass_g: body_mass_g / 1000')
 .pt.select('species', 'body_mass_g')
 .sample(10)
)

,species,body_mass_g
237,Gentoo,6.30
33,Adelie,3.90
233,Gentoo,5.85
214,Chinstrap,3.65
205,Chinstrap,4.05
69,Adelie,4.45
220,Gentoo,4.50
259,Gentoo,5.35
139,Adelie,4.25
125,Adelie,4.00


## Column names with spaces — backtick quoting
`pandas.eval()` uses **backticks**, not the single/double quotes used elsewhere in pytae, to reference a column name containing a space.

In [9]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3000.0, 4000.0], 'bill length mm': [30.0, 40.0]})
spaced.pt.mutate('bmi: `body mass g` / `bill length mm` ** 2')

,body mass g,bill length mm,bmi
0,3000.0,30.0,3.333333
1,4000.0,40.0,2.500000


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [10]:
(penguins
 .pt.mutate('bmi: body_mass_g / bill_length_mm ** 2')
 .pt.qry({'bmi': ('>', 2)})
 .pt.select('species', 'bmi')
 .sample(10)
)

,species,bmi
130,Adelie,2.243211
113,Adelie,2.400553
11,Adelie,2.589513
56,Adelie,2.333991
257,Gentoo,2.663136
309,Gentoo,2.044643
66,Adelie,2.658203
274,Gentoo,2.266158
293,Gentoo,2.404902
79,Adelie,2.256814


## Conditional column creation — `if_else()` and `case_when()`
Plain `eval()` has no ternary/`where()` support, but `mutate()` recognizes two dplyr-style expression forms by name and evaluates them via `np.where()`/`np.select()` instead: `if_else(condition, true_value, false_value)` and `case_when(cond1: val1, cond2: val2, ..., default)`. A last argument with no colon is the catch-all default (like SQL ELSE). Conditions/non-string values are still `eval()` expressions; string outcomes need quotes.

In [11]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
(penguins
 .pt.mutate("weight_class: if_else(body_mass_g > 4000, 'heavy', 'light')")
 .pt.select('species', 'body_mass_g', 'weight_class')
 .sample(10)
)

,species,body_mass_g,weight_class
71,Adelie,3900.0,light
12,Adelie,3200.0,light
328,Gentoo,4575.0,heavy
170,Chinstrap,3450.0,light
329,Gentoo,5500.0,heavy
294,Gentoo,4700.0,heavy
244,Gentoo,5000.0,heavy
212,Chinstrap,3950.0,light
257,Gentoo,5250.0,heavy
279,Gentoo,5550.0,heavy


In [12]:
# case_when(cond1: val1, cond2: val2, ..., default) — last bare value is the catch-all
# checked in order, first match wins; `True` is an optional catch-all default and must be listed last
(penguins
 .pt.mutate("size_class: case_when(body_mass_g >= 4500: 'large', body_mass_g >= 3500: 'medium', 'small')")
 .pt.select('species', 'body_mass_g', 'size_class')
 .sample(10)
)

,species,body_mass_g,size_class
158,Chinstrap,3250.0,small
25,Adelie,3800.0,medium
213,Chinstrap,3650.0,medium
15,Adelie,3700.0,medium
332,Gentoo,4650.0,large
307,Gentoo,5300.0,large
170,Chinstrap,3450.0,small
27,Adelie,3200.0,small
80,Adelie,3200.0,small
246,Gentoo,4100.0,medium
